In [ ]:
import math
from pathlib import Path
from tqdm.notebook import tqdm
from moviepy import VideoFileClip, AudioFileClip 

# Extensions map
AUDIO_EXTS = {".m4a"}

def get_unique_output_folder(base_path: Path, folder_name: str) -> Path:
    """Creates a directory, appending _v1, _v2 if it exists to avoid overwrites."""
    out_path = base_path / folder_name
    if not out_path.exists():
        out_path.mkdir(parents=True)
        return out_path
    
    counter = 1
    while True:
        new_path = base_path / f"{folder_name}_v{counter}"
        if not new_path.exists():
            new_path.mkdir(parents=True)
            return new_path
        counter += 1

In [ ]:
AUDIO_EXTS = {".m4a", ".wav"}

def segment_media(
    file_path: str | Path, 
    segment_duration: float, 
    output_base_dir: str | Path = None,
    drop_last: bool = True
):
    """
    Segments media into chunks.
    - Audio inputs (.m4a) -> Saved as .wav (PCM_16)
    - Video inputs (.mp4) -> Saved as .mp4 (Mute)
    """
    input_path = Path(file_path)
    if not input_path.exists():
        print(f" File not found: {input_path}")
        return

    # 1. Setup Paths
    is_audio = input_path.suffix.lower() in AUDIO_EXTS
    stem = input_path.stem
    output_base_dir = Path(output_base_dir) if output_base_dir else input_path.parent

    # Folder naming convention: e.g., Video1_chunks_2s
    folder_name = f"{stem}_chunks_{int(segment_duration)}s"
    save_dir = output_base_dir / folder_name
    save_dir.mkdir(parents=True, exist_ok=True)

    # 2. Load Clip
    try:
        if is_audio:
            clip = AudioFileClip(str(input_path))
        else:
            clip = VideoFileClip(str(input_path))
    except Exception as e:
        print(f"Error loading file: {e}")
        return

    duration = clip.duration
    
    # 3. Calculate Chunks
    if drop_last:
        num_chunks = math.floor(duration / segment_duration)
        drop_msg = "(Dropping last partial chunk)"
    else:
        num_chunks = math.ceil(duration / segment_duration)
        drop_msg = "(Keeping last partial chunk)"
        
    pad_len = max(3, len(str(num_chunks)))

    print(f"--- Processing: {input_path.name} ---")
    print(f"Type: {'Audio' if is_audio else 'Video'}")
    print(f"Duration: {duration:.2f}s | Segment: {segment_duration}s | Count: {num_chunks}")
    print(f"Mode: {drop_msg} | Output: {save_dir}")

    # 4. Processing Loop
    for i in tqdm(range(num_chunks), desc=f"Segmenting {stem}"):
        start_t = i * segment_duration
        end_t = min((i + 1) * segment_duration, duration)
        
        # Determine Output Filename
        if is_audio:
            # FORCE .wav extension for audio segments
            chunk_name = f"{stem}_part_{i+1:0{pad_len}d}.wav"
        else:
            # Keep original extension for video
            chunk_name = f"{stem}_part_{i+1:0{pad_len}d}{input_path.suffix}"
            
        chunk_path = save_dir / chunk_name
        
        try:
            
            subclip = clip.subclipped(start_t, end_t)
            
            if is_audio:
                # Write as standard 16-bit PCM WAV (compatible with everything)
                subclip.write_audiofile(
                    str(chunk_path), 
                    codec='pcm_s16le', 
                    logger=None
                )
            else:
                # Write video WITHOUT audio 
                subclip.write_videofile(
                    str(chunk_path),
                    codec="libx264",
                    audio=False,      
                    preset="ultrafast", 
                    threads=4,
                    logger=None
                )
        except Exception as e:
            print(f"Failed on chunk {i+1}: {e}")
        
    clip.close()
    print(f"Finished. Saved to: {save_dir}\n")

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().resolve().parents[1]))
from paths import DATA

# --- Configuration ---
VIDEO_ID = 1
SEGMENT_SEC = 10.0

# Define specific paths (Modify these two lines)
INPUT_ROOT = DATA / "Setareh/Data" 
TARGET_FILE = INPUT_ROOT / f"Video{VIDEO_ID}/Video{VIDEO_ID}.mp4"

# --- Run ---
segment_media(
    file_path=TARGET_FILE, 
    segment_duration=SEGMENT_SEC
)

In [ ]:
SEGMENT_SEC_LIST = [5.0 , 10.0]
INPUT_ROOT = DATA / "Setareh/Data" 
for MEDIA_ID in range(18):
    TARGET_VIDEO = INPUT_ROOT / f"Video{MEDIA_ID+1}/Video{MEDIA_ID+1}.mp4"
    TARGET_AUDIO = INPUT_ROOT / f"Audio{MEDIA_ID+1}/Audio{MEDIA_ID+1}.m4a"
    print(f"Processing Segment {MEDIA_ID+1}")
    for segment_duration in SEGMENT_SEC_LIST:
        segment_media(TARGET_VIDEO, segment_duration=segment_duration)
        print(f"{TARGET_VIDEO} DONE")
        segment_media(TARGET_AUDIO, segment_duration=segment_duration)
        print(f"{TARGET_AUDIO} DONE")

In [ ]:
import math
from pathlib import Path
from tqdm.notebook import tqdm
from moviepy import VideoFileClip, AudioFileClip 

# Re-define locally to add trim_end_sec logic specifically for the full runs
AUDIO_EXTS = {".m4a", ".wav"}

def segment_media_trimmed(
    file_path: str | Path, 
    segment_duration: float, 
    output_base_dir: str | Path = None,
    drop_last: bool = True,
    trim_end_sec: float = 5.0  # <--- Automatically drops the last 5s
):
    input_path = Path(file_path)
    if not input_path.exists():
        print(f" File not found: {input_path}")
        return

    is_audio = input_path.suffix.lower() in AUDIO_EXTS
    stem = input_path.stem
    output_base_dir = Path(output_base_dir) if output_base_dir else input_path.parent

    folder_name = f"{stem}_chunks_{int(segment_duration)}s"
    save_dir = output_base_dir / folder_name
    save_dir.mkdir(parents=True, exist_ok=True)

    try:
        if is_audio:
            clip = AudioFileClip(str(input_path))
        else:
            clip = VideoFileClip(str(input_path))
    except Exception as e:
        print(f"Error loading file: {e}")
        return

    # --- THE CRITICAL FIX: Pretend the file is 5s shorter ---
    duration = clip.duration - trim_end_sec 
    
    if drop_last:
        num_chunks = math.floor(duration / segment_duration)
        drop_msg = "(Dropping last partial chunk)"
    else:
        num_chunks = math.ceil(duration / segment_duration)
        drop_msg = "(Keeping last partial chunk)"
        
    pad_len = max(3, len(str(num_chunks)))

    print(f"--- Processing: {input_path.name} ---")
    print(f"Type: {'Audio' if is_audio else 'Video'}")
    print(f"Orig Duration: {clip.duration:.2f}s | Trimmed Duration: {duration:.2f}s | Count: {num_chunks}")

    for i in tqdm(range(num_chunks), desc=f"Segmenting {stem}"):
        start_t = i * segment_duration
        end_t = min((i + 1) * segment_duration, duration)
        
        if is_audio:
            chunk_name = f"{stem}_part_{i+1:0{pad_len}d}.wav"
        else:
            chunk_name = f"{stem}_part_{i+1:0{pad_len}d}{input_path.suffix}"
            
        chunk_path = save_dir / chunk_name
        
        try:
            subclip = clip.subclipped(start_t, end_t)
            if is_audio:
                subclip.write_audiofile(str(chunk_path), codec='pcm_s16le', logger=None)
            else:
                subclip.write_videofile(
                    str(chunk_path), codec="libx264", audio=False, preset="ultrafast", threads=4, logger=None
                )
        except Exception as e:
            print(f"Failed on chunk {i+1}: {e}")
    
    clip.close()
    print(f"Finished. Saved to: {save_dir}\n")


# --- RUN EXECUTION ---
SEGMENT_SEC_LIST = [1.0, 2.0, 5.0, 10.0]
INPUT_ROOT = DATA / "Setareh/runs" 

run_files = sorted(INPUT_ROOT.glob("*.mp4"))

for target_video in run_files:
    print(f"\n========== Processing Full Run: {target_video.name} ==========")
    
    target_audio = INPUT_ROOT / f"{target_video.stem}_audio.wav"
    
    # Extract full audio first if needed
    if not target_audio.exists():
        print(f"Extracting full audio from {target_video.name}...")
        try:
            clip = VideoFileClip(str(target_video))
            if clip.audio is not None:
                clip.audio.write_audiofile(str(target_audio), codec='pcm_s16le', logger=None)
            clip.close()
        except Exception as e:
            print(f"Error extracting audio from {target_video.name}: {e}")
            continue

    # Segment using the new trimmed function
    for segment_duration in SEGMENT_SEC_LIST:
        print(f"\n--- Segmenting {segment_duration}s chunks ---")
        
        segment_media_trimmed(
            file_path=target_video, 
            segment_duration=segment_duration,
            trim_end_sec=5.0
        )
        
        if target_audio.exists():
            segment_media_trimmed(
                file_path=target_audio, 
                segment_duration=segment_duration,
                trim_end_sec=5.0
            )

In [ ]:
import subprocess
import math
import csv
from pathlib import Path
from tqdm import tqdm


def segment_av(
    video_path: str | Path,
    audio_path: str | Path,
    segment_duration: float,
    output_base_dir: str | Path = None,
    drop_last: bool = True,
    skip_sec: float = None,
) -> Path:
    """Create combined audio+video .mp4 segments using ffmpeg.

    Merges a video .mp4 (video-only) with a matching audio file (.m4a/.wav),
    slicing both simultaneously so the output files contain both A+V tracks.
    These segments are what Qwen-Omni (and similar AV models) expect as input.

    skip_sec : window stride in seconds (default: segment_duration, no overlap).
               Set skip_sec < segment_duration for overlapping windows.

    Output directory : {stem}_av_chunks_{dur}s/
                       or {stem}_av_chunks_{dur}s_skip{skip}s/ when skip != dur
    Output files     : {stem}_part_{NNN}.mp4  (with audio track)
    Timing CSV       : {save_dir}/timing.csv  (part, start_sec, end_sec)
    Skips            : chunks that already exist on disk (idempotent).
    """
    if skip_sec is None:
        skip_sec = segment_duration

    video_path = Path(video_path)
    audio_path = Path(audio_path)
    output_base_dir = Path(output_base_dir) if output_base_dir else video_path.parent
    stem = video_path.stem  # e.g. "Video1"

    # ── Get video duration via ffprobe ────────────────────────────────────────
    probe = subprocess.run(
        ["ffprobe", "-v", "error",
         "-select_streams", "v:0",
         "-show_entries", "stream=duration",
         "-of", "default=noprint_wrappers=1:nokey=1",
         str(video_path)],
        capture_output=True, text=True, check=True,
    )
    raw = probe.stdout.strip()
    if not raw or raw == "N/A":
        probe2 = subprocess.run(
            ["ffprobe", "-v", "error",
             "-show_entries", "format=duration",
             "-of", "default=noprint_wrappers=1:nokey=1",
             str(video_path)],
            capture_output=True, text=True, check=True,
        )
        raw = probe2.stdout.strip()
    duration = float(raw)

    # ── Output dir ────────────────────────────────────────────────────────────
    dur_int  = int(segment_duration)
    skip_int = int(skip_sec)
    if skip_int == dur_int:
        folder_name = f"{stem}_av_chunks_{dur_int}s"
    else:
        folder_name = f"{stem}_av_chunks_{dur_int}s_skip{skip_int}s"
    save_dir = output_base_dir / folder_name
    save_dir.mkdir(parents=True, exist_ok=True)

    # sliding-window count: floor((duration - bin) / skip) + 1
    if drop_last:
        num_chunks = (int(math.floor((duration - segment_duration) / skip_sec)) + 1
                      if duration >= segment_duration else 0)
    else:
        num_chunks = int(math.ceil(duration / segment_duration))
    pad_len = max(3, len(str(num_chunks)))

    print(f"--- {stem}: {duration:.2f}s → {num_chunks} × {segment_duration}s "
          f"(skip={skip_sec}s) AV chunks ---")
    print(f"    video : {video_path.name}")
    print(f"    audio : {audio_path.name}")
    print(f"    output: {save_dir}")

    # ── Segment loop ──────────────────────────────────────────────────────────
    timing_rows = []
    for i in tqdm(range(num_chunks), desc=f"AV {stem} {dur_int}s"):
        start_t = i * skip_sec
        end_t   = start_t + segment_duration
        timing_rows.append({"part": i + 1, "start_sec": start_t, "end_sec": end_t})

        chunk_name = f"{stem}_part_{i + 1:0{pad_len}d}.mp4"
        chunk_path = save_dir / chunk_name
        if chunk_path.exists():
            continue  # idempotent

        cmd = [
            "ffmpeg", "-y",
            "-ss", str(start_t), "-t", str(segment_duration), "-i", str(video_path),
            "-ss", str(start_t), "-t", str(segment_duration), "-i", str(audio_path),
            "-c:v", "libx264", "-preset", "ultrafast",
            "-c:a", "aac", "-ar", "44100", "-ac", "2",
            "-map", "0:v:0", "-map", "1:a:0",
            "-shortest",
            str(chunk_path),
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            print(f"  ffmpeg error on chunk {i + 1}: {result.stderr[-300:]}")

    # ── Save timing CSV ───────────────────────────────────────────────────────
    timing_csv = save_dir / "timing.csv"
    with open(timing_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=["part", "start_sec", "end_sec"])
        writer.writeheader()
        writer.writerows(timing_rows)

    print(f"Done. {num_chunks} chunks → {save_dir}")
    print(f"Timing: {timing_csv}\n")
    return save_dir

In [ ]:
# ── Runner: Generate combined AV chunks for all 18 Setareh Data clips ─────────
# Uses segment_av() above to mux video + audio into single .mp4 files.
# Output lands in OUTPUT_ROOT/Video{N}/Video{N}_av_chunks_{dur}s/
# These are the files omni3b.ipynb and omni7b.ipynb expect.
#
# 2.0s added for the TopoOmni-exact Ward-linkage replication (their own paper
# convention: 2.0s non-overlapping chunks, extract_video_embeddings.py's
# chunk_duration=2.0).

INPUT_ROOT  = DATA / "segmented_stimulus/filtered"
OUTPUT_ROOT = DATA / "segmented_stimulus/filtered"
SEGMENT_DURATIONS_AV = [2.0, 5.0, 10.0]

generated_paths = []

for video_id in range(1, 19):
    video_path = INPUT_ROOT / f"Video{video_id}" / f"Video{video_id}.mp4"
    audio_path = INPUT_ROOT / f"Audio{video_id}" / f"Audio{video_id}.m4a"
    out_dir    = OUTPUT_ROOT / f"Video{video_id}"

    if not video_path.exists():
        print(f"Skipping Video{video_id}: {video_path} not found")
        continue
    if not audio_path.exists():
        print(f"Skipping Video{video_id}: {audio_path} not found")
        continue

    print(f"\n========== Video{video_id} ==========")
    for dur in SEGMENT_DURATIONS_AV:
        save_dir = segment_av(video_path, audio_path, dur, output_base_dir=out_dir)
        generated_paths.append(save_dir)

# ── Summary ───────────────────────────────────────────────────────────────────
print("\n========== Generated chunk directories ==========")
for p in generated_paths:
    print(f"  {p}")
print(f"\nTotal: {len(generated_paths)} directories.")